# ASTER — Test OOD par dégradations progressives

Ce notebook répond à la remarque suivante : le score OOD détecte-t-il spécifiquement
le boîtier ASTER, ou réagit-il plus généralement à une entrée qui s’écarte de ce qu’il
a vu pendant son développement ?

Le test part de **cellules publiques normales**, garde exactement les mêmes cellules dans
toutes les conditions, puis applique séparément :

1. un flou croissant ;
2. une altération croissante de la couleur (désaturation et décalage de teinte).

Le score OOD, le seuil, l’encodeur et les statistiques de Mahalanobis restent fixes.
Rien n’est réentraîné et aucun seuil n’est refait sur les images dégradées.

Une hausse progressive du score et du taux de rejet serait compatible avec une sensibilité
aux dégradations d’image. Elle ne démontrerait pas une détection spécifique du boîtier.


**Version locale Mac MPS, 8 Go.** Aucun réentraînement. Lots de 8 images, sauvegarde après chaque condition, reprise des calculs compatibles.

## Résultats de l'exécution locale du 25 septembre 2026

360 conditions terminées sur MPS en environ 7 minutes (calcul, hors installation).
Les mêmes 6 000 crops sont utilisés à tous les niveaux, avec modèle et seuil gelés.

| Population / altération maximale | Score médian intact → altéré | Rejets intacts → altérés |
|---|---:|---:|
| Donors (n=20), flou rayon 8 px | 467,86 → 3 649,45 | 1/20 → 20/20 |
| Donors (n=20), couleur niveau 1 | 467,86 → 571,01 | 1/20 → 9/20 |
| AML (n=10), flou rayon 8 px | 454,05 → 3 778,66 | 2/10 → 10/10 |
| AML (n=10), couleur niveau 1 | 454,05 → 578,55 | 2/10 → 5/10 |

Le score réagit aux dégradations testées, particulièrement au flou fort. La progression
n'est pas strictement croissante pour tous les patients : chez les donneurs, 11/20
trajectoires sont non décroissantes pour le flou et 13/20 pour la couleur. Le score médian
du groupe diminue légèrement entre les rayons 0 et 0,5 px. Les corrélations de rang
individuelles et les courbes détaillées figurent ci-dessous.

**Portée :** sensibilité aux altérations synthétiques dans cet échantillon public, sans
preuve de spécificité au boîtier. Ce sous-échantillon AML ne reproduit pas le taux historique
calculé sur 37 patients. Le contrôle CPU/MPS porte sur un lot de 8 crops ; l'écart de score
observé est 0,00049. Figures vérifiées visuellement et tableaux recalculés depuis les CSV.

## 1. Exécuter dans VS Code

Choisir le kernel **ASTER OOD — Mac MPS** (Python `.venv-ood/bin/python`).
Les données sont déjà présentes dans `ood_colab_inputs/` à côté de ce notebook.
Exécuter toutes les cellules. Les résultats sont écrits dans `ood_results_mps/<identifiant>/`.
Le protocole reste : 20 donneurs, 10 patients AML secondaires, 200 crops chacun,
6 niveaux de flou et 6 niveaux de couleur, seuil gelé 576,1117745962939.
Les mêmes crops sont utilisés partout, sans sélection selon le score OOD.
Une session correspond ici à un patient public. Aucun diagnostic clinique n'est calculé.


In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path(os.environ.get("ASTER_REPO_ROOT", Path.cwd())).resolve()
CORPUS_ROOT = Path(os.environ.get("ASTER_OOD_CORPUS", REPO_ROOT / "external_data/ood_caitomorph/corpus"))
MANIFEST_PATH = Path(os.environ.get("ASTER_OOD_MANIFEST", CORPUS_ROOT / "manifest.csv"))
HEADS_PATH = REPO_ROOT / "models/mil/heads.pt"
ENCODER_PATH = None
OOD_STATS_PATH = REPO_ROOT / "models/mil/ood_stats.npz"
NOTEBOOK_PATH = REPO_ROOT / "experiments/ood_progressive_degradation/ood_progressive_degradation_mps.ipynb"
RESULTS_ROOT = REPO_ROOT / "experiments/ood_progressive_degradation/results/reproduction"
SEED = 42
N_DONOR_SESSIONS = 20
N_AML_SESSIONS = 10
CROPS_PER_SESSION = 200
BATCH_SIZE = 8
DEVICE = "mps"
EXPECTED_FROZEN_THRESHOLD = 576.1117745962939


## 2. Vérifier MPS et les fichiers locaux

In [ ]:
import sys, json, hashlib, random, csv, time, math, gc, os, platform
from datetime import datetime, timezone
import numpy as np, pandas as pd, torch, torchvision, PIL, scipy
from PIL import Image, ImageEnhance, ImageFilter
from scipy.stats import spearmanr
from torch import nn
from torchvision import transforms, models
import matplotlib.pyplot as plt
from IPython.display import display

assert torch.backends.mps.is_available(), "MPS indisponible : choisir le kernel .venv-ood."
torch.set_num_threads(2)
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
for path in (MANIFEST_PATH, HEADS_PATH, OOD_STATS_PATH):
    assert path.is_file(), f"Fichier absent : {path}"
print("Appareil :", DEVICE, "| Python :", platform.python_version(), "| Torch :", torch.__version__)
print("Lot :", BATCH_SIZE, "images | Corpus :", CORPUS_ROOT)


## 3. Charger le manifest et constituer les bags fixes

La sélection est faite avant les dégradations et ne dépend jamais du score OOD. Une session
correspond ici à un patient public. Pour chaque patient, le tirage aléatoire de 200 cellules
est effectué une seule fois ; la liste est ensuite réutilisée pour le niveau intact et tous
les niveaux dégradés.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
required = {"source", "patient_id", "corpus_path"}
missing = required - set(manifest.columns)
assert not missing, f"Colonnes manquantes dans manifest.csv : {sorted(missing)}"

def row_label(row):
    if "bag_label" in row and pd.notna(row["bag_label"]):
        return str(row["bag_label"])
    if "diagnosis_fine" in row and pd.notna(row["diagnosis_fine"]):
        return str(row["diagnosis_fine"])
    if "extra" in row and pd.notna(row["extra"]):
        try:
            extra = json.loads(row["extra"])
            return str(extra.get("diagnosis_fine", extra.get("bag_label", "")))
        except (TypeError, json.JSONDecodeError):
            return ""
    return ""

manifest["_label"] = manifest.apply(row_label, axis=1)
manifest["_path"] = manifest["corpus_path"].map(lambda p: CORPUS_ROOT / str(p))
assert manifest["_path"].map(Path.is_file).all(), "Au moins une image du manifest est absente."
cai = manifest[manifest.source.eq("caitomorph")].copy()
assert len(cai) > 0, "Aucune ligne cAItomorph dans le manifest."

def fixed_sessions(label, count):
    groups=[]
    for patient, group in sorted(cai[cai._label.eq(label)].groupby("patient_id"), key=lambda x: x[0]):
        group = group.sort_values("corpus_path")
        if len(group) >= CROPS_PER_SESSION:
            groups.append((patient, group))
    assert len(groups) >= count, f"Seulement {len(groups)} sessions {label} avec {CROPS_PER_SESSION} cellules disponibles."
    rng = random.Random(SEED + sum(map(ord, label)))
    selected=[]
    for patient, group in rng.sample(groups, count):
        indices = sorted(rng.sample(range(len(group)), CROPS_PER_SESSION))
        selected.append({"patient_id": patient, "diagnosis": label,
                         "paths": group.iloc[indices]["_path"].tolist(),
                         "path_sha256": hashlib.sha256("\n".join(map(str, group.iloc[indices]["_path"].tolist())).encode()).hexdigest()})
    return selected

sessions = fixed_sessions("Stem cell donor", N_DONOR_SESSIONS) + fixed_sessions("AML", N_AML_SESSIONS)
assert len({s["patient_id"] for s in sessions}) == len(sessions)
print(pd.DataFrame([{"diagnosis":s["diagnosis"], "sessions":sum(x["diagnosis"]==s["diagnosis"] for x in sessions), "crops":len(s["paths"])} for s in sessions]).drop_duplicates().to_string(index=False))

## 4. Verrouiller l’encodeur et la gate de l’article

La moyenne, la précision et le seuil sont chargés tels quels depuis le fichier historique.
Le notebook refuse avec une erreur explicite toute autre valeur de seuil. Il ne charge pas les fichiers
post-amendement et ne recalcule jamais les statistiques.

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()

gate = np.load(OOD_STATS_PATH)
for key in ("mean", "precision", "threshold"):
    assert key in gate.files, f"Clé absente des statistiques OOD : {key}"
mean = np.asarray(gate["mean"], dtype=np.float32)
precision = np.asarray(gate["precision"], dtype=np.float32)
threshold = float(gate["threshold"])
assert mean.shape == (512,) and precision.shape == (512,512)
assert abs(threshold - EXPECTED_FROZEN_THRESHOLD) < 1e-6, f"Threshold non gelé : {threshold}"

if HEADS_PATH is not None:
    checkpoint = torch.load(HEADS_PATH, map_location="cpu", weights_only=True)
    assert isinstance(checkpoint, dict) and "encoder" in checkpoint, "heads.pt ne contient pas encoder."
    encoder_state = checkpoint["encoder"]
else:
    checkpoint = torch.load(ENCODER_PATH, map_location="cpu", weights_only=True)
    encoder_state = checkpoint.get("encoder", checkpoint) if isinstance(checkpoint, dict) else checkpoint

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=None)
        self.stem = nn.Sequential(*list(backbone.children())[:-1])
    def forward(self, images):
        return self.stem(images).flatten(1)
encoder = Encoder()
encoder.load_state_dict(encoder_state, strict=True)
encoder.eval().to(DEVICE)

IMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)
class SquarePad:
    def __call__(self, image):
        image=image.convert("RGB"); w,h=image.size; side=max(w,h)
        left=(side-w)//2; top=(side-h)//2
        right=side-w-left; bottom=side-h-top
        corners=np.asarray([image.getpixel((0,0)), image.getpixel((max(0,w-1),0)),
                            image.getpixel((0,max(0,h-1))), image.getpixel((max(0,w-1),max(0,h-1)))])
        fill=tuple(np.median(corners,axis=0).astype(np.uint8).tolist())
        from PIL import ImageOps
        return ImageOps.expand(image,border=(left,top,right,bottom),fill=fill)
transform=transforms.Compose([SquarePad(), transforms.Resize((224,224)), transforms.ToTensor(),
                              transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
print("Frozen OOD threshold:", threshold, "| stats:", sha256_file(OOD_STATS_PATH))

MODEL_PATH = HEADS_PATH or ENCODER_PATH
print("Model bundle:", MODEL_PATH)


## 5. Dégradations déterministes

Les niveaux sont définis avant de regarder les résultats. Le niveau 0 est l’image intacte.
Le flou et la couleur sont testés séparément afin de savoir quel type de dégradation est
responsable d’un éventuel changement. L’image de chaque cellule reste la même d’un niveau
à l’autre ; seule la transformation change.

In [ ]:
BLUR_LEVELS = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]       # rayon GaussianBlur en pixels
COLOR_LEVELS = [0.0, 0.10, 0.25, 0.50, 0.75, 1.0]    # désaturation + décalage de teinte

def degrade_blur(image, level):
    return image if level == 0 else image.filter(ImageFilter.GaussianBlur(radius=float(level)))

def degrade_color(image, level):
    if level == 0: return image
    image = ImageEnhance.Color(image).enhance(max(0.0, 1.0-float(level)))
    hsv = np.array(image.convert("HSV"), dtype=np.uint16, copy=True)
    shift = int(round(90.0 * float(level)))
    hsv[...,0] = (hsv[...,0] + shift) % 256
    return Image.fromarray(hsv.astype(np.uint8), mode="HSV").convert("RGB")

FAMILIES = {"blur": (BLUR_LEVELS, degrade_blur), "color": (COLOR_LEVELS, degrade_color)}

## 6. Calculer le score OOD sans réentraînement

Le score est exactement la médiane des distances de Mahalanobis par crop :

`median((h - mean)^T precision (h - mean))`.

Le seuil de rejet reste celui de l’article. Pour chaque condition, les 200 crops du patient
sont encodés et agrégés en une seule session. Aucun score clinique ou classement de maladie
n’est calculé ici.

Avant le calcul complet, un lot intact est comparé sur CPU et MPS. Le contrôle est numérique, sans ajuster le modèle ou le seuil. Le niveau intact est calculé une fois par patient puis réutilisé pour les deux familles.

In [ ]:
# Protocole écrit AVANT les calculs ; il protège la reprise contre un changement des entrées.
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()

selection = [{"patient_id": s["patient_id"], "diagnosis": s["diagnosis"],
              "paths": [str(p.relative_to(CORPUS_ROOT)) for p in s["paths"]]} for s in sessions]
image_hashes = {p: sha256_file(CORPUS_ROOT / p) for s in selection for p in s["paths"]}
protocol = {"schema": "aster-ood-mps-v1", "seed": SEED,
    "n_donor_sessions": N_DONOR_SESSIONS, "n_aml_sessions": N_AML_SESSIONS,
    "crops_per_session": CROPS_PER_SESSION, "batch_size": BATCH_SIZE, "device": DEVICE,
    "families": {"blur": BLUR_LEVELS, "color": COLOR_LEVELS}, "threshold": threshold,
    "heads_sha256": sha256_file(MODEL_PATH), "ood_stats_sha256": sha256_file(OOD_STATS_PATH),
    "manifest_sha256": sha256_file(MANIFEST_PATH), "selection": selection,
    "image_content_sha256": digest(image_hashes),
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
        "torchvision": torchvision.__version__, "numpy": np.__version__,
        "pandas": pd.__version__, "pillow": PIL.__version__, "scipy": scipy.__version__},
    "refit": False, "threshold_tuning": False}
notebook_content = json.loads(NOTEBOOK_PATH.read_text())
protocol["computation_sha256"] = digest([c["source"] for c in notebook_content["cells"] if c["cell_type"] == "code"])
OUTPUT_ROOT = RESULTS_ROOT / digest(protocol)[:16]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "protocol.json").write_text(json.dumps(protocol, indent=2, default=str))
(OUTPUT_ROOT / "image_hashes.json").write_text(json.dumps(image_hashes, indent=2))

@torch.inference_mode()
def parity_check():
    images = []
    for p in sessions[0]["paths"][:BATCH_SIZE]:
        with Image.open(p) as im:
            images.append(transform(im.convert("RGB")))
    tensor = torch.stack(images)
    cpu_encoder = Encoder().eval()
    cpu_encoder.load_state_dict(encoder_state, strict=True)
    cpu = cpu_encoder(tensor).numpy()
    mps = encoder(tensor.to(DEVICE)).cpu().numpy()
    assert np.isfinite(mps).all()
    np.testing.assert_allclose(mps, cpu, rtol=1e-3, atol=1e-4)
    def score(h):
        d = h - mean
        return float(np.median(np.einsum("ij,jk,ik->i", d, precision, d)))
    c, m = score(cpu), score(mps)
    assert abs(c-m) <= max(0.1, abs(c)*1e-3), (c, m)
    result = {"n_crops": len(images), "feature_max_absolute_error": float(np.abs(cpu-mps).max()),
              "cpu_score": c, "mps_score": m, "absolute_score_error": abs(c-m)}
    (OUTPUT_ROOT / "cpu_mps_parity.json").write_text(json.dumps(result, indent=2))
    print("Contrôle CPU/MPS :", result, flush=True)
    del cpu_encoder, tensor
    gc.collect()
    torch.mps.empty_cache()

parity_check()

@torch.inference_mode()
def session_score(paths, transform_fn, level):
    distances = []
    for start in range(0, len(paths), BATCH_SIZE):
        batch = []
        for path in paths[start:start+BATCH_SIZE]:
            with Image.open(path) as im:
                batch.append(transform(transform_fn(im.convert("RGB"), level)))
        tensor = torch.stack(batch).to(DEVICE)
        features = encoder(tensor).cpu().numpy().astype(np.float32)
        delta = features - mean
        # Même formule et même précision que le notebook Colab.
        distances.extend(np.einsum("ij,jk,ik->i", delta, precision, delta).tolist())
        del tensor, features, delta, batch
    assert len(distances) == len(paths) and np.isfinite(distances).all()
    score = float(np.median(distances))
    return score, bool(score > threshold)

cache_path = OUTPUT_ROOT / "progress.json"
rows = json.loads(cache_path.read_text()) if cache_path.is_file() else []
done = {(r["session_id"], r["family"], r["level_index"]): r for r in rows}
total = len(sessions) * sum(len(levels) for levels, fn in FAMILIES.values())
started = time.monotonic()
initial_count = len(rows)
print("Résultats :", OUTPUT_ROOT, "| Déjà terminées :", len(rows), "/", total, flush=True)
for family, (levels, fn) in FAMILIES.items():
    for s in sessions:
        for index, level in enumerate(levels):
            key = (s["patient_id"], family, index)
            if key in done:
                continue
            baseline = next((r for r in rows if r["session_id"] == s["patient_id"] and r["level_index"] == 0), None)
            if index == 0 and baseline is not None:
                score, rejected = baseline["ood_score"], baseline["rejected"]
            else:
                score, rejected = session_score(s["paths"], fn, level)
            row = dict(session_id=s["patient_id"], diagnosis=s["diagnosis"], family=family,
                level_index=index, level=float(level), ood_score=score, threshold=threshold,
                rejected=rejected, n_crops=len(s["paths"]), path_sha256=s["path_sha256"])
            rows.append(row)
            done[key] = row
            temp = cache_path.with_suffix(".tmp")
            temp.write_text(json.dumps(rows, indent=2))
            temp.replace(cache_path)
            pd.DataFrame(rows).to_csv(OUTPUT_ROOT / "ood_scores_partial.csv", index=False)
            elapsed = time.monotonic() - started
            eta = elapsed / max(1, len(rows)-initial_count) * (total-len(rows))
            status = {"completed": len(rows), "total": total, "elapsed_seconds_this_run": elapsed,
                      "estimated_remaining_seconds": eta, "updated_utc": datetime.now(timezone.utc).isoformat()}
            (OUTPUT_ROOT / "status.json").write_text(json.dumps(status, indent=2))
            if len(rows) % 6 == 0:
                print(f"{len(rows)}/{total} | {family} | {s['patient_id']} | score {score:.2f} | reste ~{eta/60:.1f} min", flush=True)
                torch.mps.empty_cache()
scores = pd.DataFrame(rows)
assert len(scores) == total and not scores.duplicated(["session_id", "family", "level_index"]).any()
scores.to_csv(OUTPUT_ROOT / "ood_scores.csv", index=False)
print("Calcul complet.", flush=True)


## 7. Résumer la progression et la monotonicité

Une corrélation de Spearman positive indique que le score tend à monter avec la dégradation.
La colonne `monotonic_non_decreasing` est plus stricte : elle exige que les six niveaux ne
descendent jamais pour une session. Les deux mesures sont rapportées par groupe, sans cacher
les sessions qui ne suivent pas la tendance.

In [ ]:
def summarize(frame):
    out=[]
    for (diagnosis,family), group in frame.groupby(["diagnosis","family"]):
        per=[]
        for sid, session in group.groupby("session_id"):
            session=session.sort_values("level_index")
            rho=float(spearmanr(session.level_index,session.ood_score).statistic)
            per.append(dict(session_id=sid, spearman_rho=rho,
                           monotonic_non_decreasing=bool(np.all(np.diff(session.ood_score)>=-1e-8)),
                           score_level0=float(session.ood_score.iloc[0]), score_last=float(session.ood_score.iloc[-1]),
                           rejected_level0=bool(session.rejected.iloc[0]), rejected_last=bool(session.rejected.iloc[-1])))
        per=pd.DataFrame(per)
        medians=group.groupby("level_index").ood_score.median().to_dict()
        rates=group.groupby("level_index").rejected.mean().to_dict()
        out.append(dict(diagnosis=diagnosis,family=family,n_sessions=len(per),
            median_score_level0=medians[0],median_score_last=medians[max(medians)],
            median_score_delta=medians[max(medians)]-medians[0],
            rejection_rate_level0=rates[0],rejection_rate_last=rates[max(rates)],
            median_spearman_rho=float(per.spearman_rho.median()),
            fraction_monotonic_non_decreasing=float(per.monotonic_non_decreasing.mean())))
        per.to_csv(OUTPUT_ROOT/f"per_session_{diagnosis.replace(' ','_')}_{family}.csv",index=False)
    return pd.DataFrame(out)

OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
scores.to_csv(OUTPUT_ROOT/"ood_scores.csv",index=False)
summary=summarize(scores)
summary.to_csv(OUTPUT_ROOT/"ood_summary.csv",index=False)
display(summary.round(4))

## 8. Courbes et interprétation

La bande montre l'intervalle interquartile entre patients, pas un intervalle de confiance.
Les donneurs et les patients AML restent séparés. Les niveaux de couleur combinent
désaturation et décalage de teinte : ils ne permettent pas d'isoler ces deux effets.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), layout="constrained")
for column, (family, (levels, fn)) in enumerate(FAMILIES.items()):
    frame = scores[scores.family.eq(family)]
    x = np.asarray(levels)
    for diagnosis, color, marker, style in [("Stem cell donor", "#0072B2", "o", "-"), ("AML", "#D55E00", "s", "--")]:
        sub = frame[frame.diagnosis.eq(diagnosis)]
        grouped = sub.groupby("level_index").ood_score
        median, q25, q75 = grouped.median(), grouped.quantile(.25), grouped.quantile(.75)
        n = sub.session_id.nunique()
        label = f"{'Donors' if diagnosis == 'Stem cell donor' else 'AML'} (n={n})"
        axes[0, column].plot(x, median, marker=marker, linestyle=style, label=label, color=color)
        axes[0, column].fill_between(x, q25, q75, alpha=.15, color=color)
        rates = sub.groupby("level_index").rejected.mean() * 100
        axes[1, column].plot(x, rates, marker=marker, linestyle=style, label=label, color=color)
    axes[0, column].axhline(threshold, color="black", linestyle=":", label=f"Threshold {threshold:.2f}")
    axes[0, column].set_title("Gaussian blur" if family == "blur" else "Color perturbation")
    axes[0, column].set_ylabel("Median bag OOD score (IQR)")
    axes[1, column].set_ylabel("Rejected patients (%)")
    axes[1, column].set_ylim(-2, 102)
    for row in range(2):
        ax = axes[row, column]
        ax.set_xlabel("Blur radius (pixels)" if family == "blur" else "Color perturbation level")
        ax.grid(axis="y", alpha=.2)
        ax.legend(fontsize=9)
fig.suptitle("OOD sensitivity to progressive degradation — frozen model and threshold")
fig.savefig(OUTPUT_ROOT / "ood_progressive_degradation.png", dpi=180)
fig.savefig(OUTPUT_ROOT / "ood_progressive_degradation.pdf")
plt.show()

parts = []
for row in summary.itertuples():
    parts.append(f"{row.diagnosis} / {row.family} : médiane {row.median_score_level0:.2f} → "
                 f"{row.median_score_last:.2f}, rejets {row.rejection_rate_level0:.0%} → "
                 f"{row.rejection_rate_last:.0%}, rho médian {row.median_spearman_rho:.3f}, "
                 f"sessions non décroissantes {row.fraction_monotonic_non_decreasing:.0%}.")
interpretation = "\n".join(parts) + "\n\nUne hausse finale ne suffit pas à établir la monotonie. " \
    "Rapporter les courbes et leurs exceptions. Ce test mesure la sensibilité aux transformations " \
    "synthétiques sur des crops publics ; il ne démontre pas une détection spécifique du boîtier."
print(interpretation)


## 9. Archiver la provenance et les limites

Le fichier `protocol.json` enregistre les hashes des modèles, du fichier OOD, du manifest et
des listes de crops. Le fichier `interpretation.md` ne transforme pas automatiquement un
résultat en preuve de spécificité : il rappelle exactement ce que l’expérience permet de dire.

Limites à conserver dans l’article : une seule configuration OOD, un nombre limité de
patients publics, des dégradations synthétiques sur des crops déjà extraits, et aucune
preuve que le score identifie la caméra plutôt qu’un changement de distribution. Une étude
multimicroscope avec images réellement acquises sur plusieurs boîtiers serait nécessaire
pour revendiquer cette spécificité.

In [ ]:
(OUTPUT_ROOT / "interpretation.md").write_text("# Résultats du test OOD local\n\n" + interpretation + "\n")
print("Résultats complets :", OUTPUT_ROOT)
print("Le protocole, les hashes, le contrôle CPU/MPS, les CSV et les figures sont archivés.")


## Conclusion à reporter si les résultats la soutiennent

> The OOD gate was not interpreted as a camera identifier. Using the frozen article
> configuration, public donor cell bags were subjected to predefined, progressively stronger
> blur and colour perturbations while the crop identities and gate threshold were held fixed.
> The resulting score and rejection-rate curves quantify sensitivity to these image shifts.
> This experiment supports degradation sensitivity if the curves are monotone, but it does
> not establish acquisition-device specificity.